# Discover a recruitment model from the AI Act

You are building a recruitment application. Which roles and relationships does the source
support? Discover them from the complete pinned English Official Journal PDF, then inspect the
evidence behind each draft. All proposals stay pending for review.

From a clone, install the notebook environment.

```sh
uv sync --locked --group notebook --extra documents --extra inference --extra discovery-workflow
```

 Put `OPENAI_API_KEY` in the repository-root `.env`, select the project kernel and **Run
All**. The first run downloads the PDF as needed and makes multiple paid Luna calls. Restart
in the same workspace to reopen the result. The local stage journal contains source quotations
and provider responses.

In [2]:
import json
import sys
from pathlib import Path

root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()
)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from domain_kernel import PDF, DocumentSpan
from examples.notebooks.document_discovery import configured_client, open_recruitment_model
from examples.notebooks.originals import load_originals

workspace = root / "local/staged-recruitment-discovery"
originals = root / "local/notebook-tour/recruitment-default-discovery/sources"
original = load_originals(directory=originals, identifiers=["ai-act-en"], download=True)[
    "ai-act-en"
]
client, budget, graph = configured_client(workspace=workspace, originals=originals)
actor = "demo:discovery-reader"
project, model = await open_recruitment_model(client=client, actor=actor)
print(model.info.description)
print(original.digest.value)

Describe how AI systems are used in recruitment and selection, who is involved, and their responsibilities.
sha256:bba630444b3278e881066774002a1d7824308934f49ccfa203e65be43692f55e


## Read the source

Docling groups native document items by structure and token budget. The SDK retains those
chunks and separately addressable page text in SQLite. Read an original page for comparison.

In [3]:
source = await project.ingest_source(
    name="english-ai-act",
    source=PDF(path=original.relative_path, expected_hash=original.digest),
    actor=actor,
    idempotency_key="ingest-v1",
)
pages = {
    e.locator.page: e
    for e in source.evidence
    if isinstance(e.locator, DocumentSpan) and e.locator.section is None
}
print(f"{len(pages)} physical pages")
print((await project.get_evidence(evidence=pages[127])).excerpt)

144 physical pages
ANNEX III
High-risk AI systems referred to in Article 6(2)
High-risk AI systems pursuant to Article 6(2) are the AI systems listed in any of the following areas:
1. Biometrics, in so far as their use is permitted under relevant Union or national law:
(a) remote biometric identification systems.
This shall not include AI systems intended to be used for biometric verification the sole purpose of which is to
confirm that a specific natural person is the person he or she claims to be;
(b) AI systems intended to be used for biometric categorisation, according to sensitive or protected attributes or
characteristics based on the inference of those attributes or characteristics;
(c) AI systems intended to be used for emotion recognition.
2. Critical infrastructure: AI systems intended to be used as safety components in the management and operation of
critical digital infrastructure, road traffic, or in the supply of water, gas, heating or electricity.
3.
Education and vocati

## Discover and verify

Pass the revision to the SDK. The executor extracts claims, checks omissions, verifies their
quotations and meaning, and reviews relationship endpoints after resolving aliases. The model
description supplies the recruitment scope.

In [4]:
before = budget.usage()
run = await model.discover(
    sources=[source.revision], actor=actor, idempotency_key="discover-verified-v3"
)
if run.failures:
    raise RuntimeError([(f.code, f.issue.path, f.message) for f in run.failures])
cases = [case async for case in model.iter_review_cases(run_id=run.id)]
print(f"{len(cases)} review cases · {run.status.value}")
print(f"New provider attempts this execution: {budget.usage().attempts - before.attempts}")

45 review cases · needs_review
New provider attempts this execution: 0


## Inspect the discovered model

Each row is a proposal from this run. Follow the citations to check the meaning, including any
conditions or exceptions. Review findings include statements the verifier withheld.

In [5]:
audit = json.loads((graph.directory / f"{run.id.value}.json").read_bytes())
for claim in audit["claims"]:
    print(f"\n{claim['kind']} · {claim['title']}")
    if claim["kind"] == "relationship":
        print(f"{claim['source']} → {claim['target']}")
    print(claim["meaning"])
    for part in [claim, *claim["conditions"], *claim["exceptions"]]:
        if "text" in part:
            print(part["text"])
        for anchor in part["evidence"]:
            locator = audit["locators"][anchor["citation"] - 1]["locator"]
            print(f"  {locator} · {anchor['quote']}")


concept · Deployer
A natural or legal person, including a public authority, agency or other body, that uses an AI system under its authority.
  {'start': 0, 'end': 5438, 'page': 4, 'section': None, 'kind': 'document_span'} · any natural or legal person, including a public authority, agency or other body, using an AI system under its authority
The AI system is not used in the course of a personal non-professional activity.
  {'start': 0, 'end': 5438, 'page': 4, 'section': None, 'kind': 'document_span'} · except where the AI system is used in the course of a personal non-professional activity

relationship · Deployer uses AI system
deployer → ai-system
A deployer uses an AI system under the deployer's authority.
  {'start': 0, 'end': 5438, 'page': 4, 'section': None, 'kind': 'document_span'} · using an AI system under its authority
The AI system is not used in the course of a personal non-professional activity.
  {'start': 0, 'end': 5438, 'page': 4, 'section': None, 'kind': 'document_sp

In [6]:
from collections import Counter

print("Findings by review stage", dict(Counter(item["stage"] for item in audit["findings"])))
for component in audit["components"]:
    print(", ".join(component))

Findings by review stage {'lineage': 15, 'semantic': 101, 'reconcile': 24, 'endpoints': 17, 'connectivity': 1}
affected-person, ai-literacy, ai-system, ai-system-for-recruitment-and-selection, deployer, deployer-controlled-system-log, high-risk-ai-system, input-data, natural-person-category-or-group-likely-affected, natural-person-human-oversight-assignee, natural-person-subject, provider
affected-workers, employer, workers-representatives
ai-recruitment-or-selection-system, natural-person-candidate
distributor
employee
employment-ai-use-case
fundamental-rights-impact-assessment
importer
market-surveillance-authority
person-subject-to-recruitment-and-selection
platform-service-provider
relevant-stakeholder
staff-and-other-persons-dealing-with-ai
whistleblower
